# StatCan parser walkthrough

This notebook is the practical guide for parsing Statistics Canada supply-use and symmetric input-output tables in MARIO.


## What this notebook covers

- where the StatCan tables come from;
- when direct online parsing is enough and when a local cache is useful;
- how `SUT` and `IOT` workflows differ;
- how `year=`, `level=`, `geo=`, and `valuation=` are used;
- what to expect from `download=True`;
- which caveats matter for this API-driven parser.


## Relevant source pages

- Official StatCan SUT catalogue: [Supply and use tables](https://www150.statcan.gc.ca/n1/en/catalogue/36100478)
- Official StatCan IOT catalogue: [Symmetric input-output tables](https://www150.statcan.gc.ca/n1/en/catalogue/36100001)
- StatCan WDS documentation: [WDS user guide](https://www.statcan.gc.ca/en/developers/wds/user-guide)

MARIO can query the official WDS API directly, so there is no need to download the raw `.csv` bundles manually unless you want a local cache.


## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_statcan(...)`

The same function supports:

- `SUT` tables;
- `IOT` tables;
- `summary`, `detail`, and `link1997` levels where available;
- direct online parsing or local cached raw files.


## Key arguments

The key public arguments are:

- `year`: reference year to parse;
- `table`: choose `"SUT"` or `"IOT"`;
- `level`: choose `summary`, `detail`, or `link1997` when supported;
- `geo`: geography label such as `Canada` or one province/territory;
- `valuation`: only for `IOT`, usually `basic` or `purchaser`;
- `path`: optional directory for locally cached raw files;
- `download`: when `True`, MARIO downloads the raw WDS bundle into `path` and then parses it locally.


## `SUT` versus `IOT`

Use `table="SUT"` when you want the native split structure with `S`, `U`, `Yc`, `Ya`, `Va`, and `Vc`.

Use `table="IOT"` when you want the symmetric input-output representation with `Z`, `Y`, and `V`.

For `SUT`, `level="link1997"` is also available. For `IOT`, `valuation=` matters because the tables are exposed at both basic and purchaser prices.


## Direct online parsing versus local cache

Use direct online parsing when you just want the database and do not need the raw files afterwards.

Use `download=True` together with `path=...` when you want MARIO to keep the raw WDS `.csv` bundle locally. This is the most useful option when you parse the same StatCan table repeatedly or want a reproducible local cache.


In [ ]:
import mario

## Parse a Canada `SUT` directly from WDS

This is the simplest StatCan workflow.


In [ ]:
db = mario.parse_statcan(
    year=2022,
    table="SUT",
    level="summary",
    geo="Canada",
    calc_all=False,
)

db

## Parse a provincial `SUT`

The `SUT` tables expose provinces and territories as separate geographies. The parser expects the published StatCan geography label.


In [ ]:
db = mario.parse_statcan(
    year=2022,
    table="SUT",
    level="detail",
    geo="Ontario",
    calc_all=False,
)

db

## Parse an `IOT` at basic prices

For `IOT`, pass `valuation="basic"` or `valuation="purchaser"`.


In [ ]:
db = mario.parse_statcan(
    year=2022,
    table="IOT",
    level="detail",
    valuation="basic",
    calc_all=False,
)

db

## Parse an `IOT` at purchaser prices

The public API stays the same; only the `valuation` selector changes.


In [ ]:
db = mario.parse_statcan(
    year=2022,
    table="IOT",
    level="summary",
    valuation="purchaser",
    calc_all=False,
)

db

## Keep the raw WDS files locally

When `download=True`, MARIO stores the downloaded raw `.csv` bundle inside `path` and then parses the local file. This is useful when you want to avoid re-downloading the same table.


In [ ]:
db = mario.parse_statcan(
    year=2022,
    table="SUT",
    level="detail",
    geo="Quebec",
    path="/path/to/statcan_cache",
    download=True,
    calc_all=False,
)

db

## First inspection

Once the database is parsed, the normal MARIO inspection helpers work exactly as for the other parsers.


In [ ]:
db.table_type
db.sets
db.get_index("Region")
db.units

## Caveats

- the parser is API-driven, so network availability matters more than for local-file parsers;
- `download=True` is optional, not required;
- this walkthrough focuses on the economic StatCan tables only;
- environmental extensions are intentionally left out here.
